# A5 Transformer arithmetic — core

Adapted from University of Michigan EECS 498/598, Deep Learning for Computer Vision (Winter 2022), by Justin Johnson and course staff.

**Static validation only; not executed.** Enable the run switches explicitly; generated files go to `core_results/`.

In [ ]:
from pathlib import Path
import os
import sys
import torch
import matplotlib.pyplot as plt
candidates = []
for base in (Path.cwd(), *Path.cwd().parents,
             Path('/content/drive/MyDrive/umich-dlcv-wi2022')):
    candidates.extend((base, base / 'assignments' / 'a5'))
ASSIGNMENT_DIR = next((p.resolve() for p in candidates if (p / 'transformers_core.py').is_file()), None)
if ASSIGNMENT_DIR is None:
    raise FileNotFoundError('Open from the project/assignment directory, or mount the canonical Colab Drive project first.')
loaded_helpers = sys.modules.get('eecs598')
if loaded_helpers is not None:
    loaded_path = Path(loaded_helpers.__file__).resolve().parent
    if loaded_path != (ASSIGNMENT_DIR / 'eecs598').resolve():
        raise RuntimeError('Restart the kernel before switching assignments: eecs598 is already loaded from another assignment.')
sys.path.insert(0, str(ASSIGNMENT_DIR))
os.chdir(ASSIGNMENT_DIR)
from eecs598.utils import reset_seed
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR = ASSIGNMENT_DIR / 'core_results' / 'transformers'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_OVERFIT = False
ALLOW_DOWNLOAD = False
reset_seed(0)
plt.rcParams['image.cmap'] = 'gray'
print('Outputs:', OUTPUT_DIR)

In [ ]:
from sklearn.model_selection import train_test_split
from transformers_core import (Transformer, AddSubDataset, generate_token_dict,
    prepocess_input_sequence, position_encoding_simple, position_encoding_sinusoid,
    CrossEntropyLoss, LabelSmoothingLoss)
from a5_helper_core import get_toy_data, train as train_transformer, val as val_transformer, inference, draw
SPECIAL_TOKENS = ['POSITIVE', 'NEGATIVE', 'add', 'subtract', 'BOS', 'EOS']
vocab = list('0123456789') + SPECIAL_TOKENS
token_dict = generate_token_dict(vocab)
BATCH_SIZE, EMB_DIM, NUM_HEADS, LAYERS = 256, 64, 4, 2
def make_model():
    return Transformer(NUM_HEADS, EMB_DIM, 128, 0.1, LAYERS, LAYERS, len(vocab)).to(DEVICE)

## Arithmetic data
Use the ordinary notebook’s 90/10 seed-0 split. Simple position encoding is retained in the optional small overfit; full training uses sinusoidal encoding.

In [ ]:
if RUN_TRAINING or RUN_EVALUATION or RUN_OVERFIT:
    data = get_toy_data(str(ASSIGNMENT_DIR / 'two_digit_op.json'))
    X_train, X_val, y_train, y_val = train_test_split(data['inp_expression'], data['out_expression'], test_size=0.1, random_state=0)
    train_data = AddSubDataset(X_train, y_train, token_dict, SPECIAL_TOKENS, EMB_DIM, position_encoding_sinusoid)
    valid_data = AddSubDataset(X_val, y_val, token_dict, SPECIAL_TOKENS, EMB_DIM, position_encoding_sinusoid)
    train_loader = torch.utils.data.DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)
    valid_loader = torch.utils.data.DataLoader(valid_data, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

In [ ]:
if RUN_OVERFIT:
    reset_seed(0)
    overfit_data = AddSubDataset(X_train, y_train, token_dict, SPECIAL_TOKENS, 32, position_encoding_simple)
    indices = torch.linspace(0, len(overfit_data) - 1, steps=4).long()
    small_loader = torch.utils.data.DataLoader(torch.utils.data.Subset(overfit_data, indices), batch_size=4, shuffle=False)
    small_model = Transformer(4, 32, 32, 0.2, 1, 1, len(vocab)).to(DEVICE)
    train_transformer(small_model, small_loader, small_loader, CrossEntropyLoss,
                      num_epochs=200, lr=1e-3, batch_size=4, warmup_interval=None, device=DEVICE)
    print('Overfit token accuracy:', val_transformer(small_model, small_loader, CrossEntropyLoss, 4, device=DEVICE)[1])

## Full training
60 epochs at 0.001 followed by 40 refinement epochs at 0.0003, matching the ordinary notebook.

In [ ]:
if RUN_TRAINING:
    reset_seed(0)
    model = make_model()
    model = train_transformer(model, train_loader, valid_loader, CrossEntropyLoss, 60,
                              lr=1e-3, batch_size=BATCH_SIZE, warmup_interval=None, device=DEVICE)
    model = train_transformer(model, train_loader, valid_loader, CrossEntropyLoss, 40,
                              lr=3e-4, batch_size=BATCH_SIZE, warmup_interval=None, device=DEVICE)
    torch.save(model.state_dict(), OUTPUT_DIR / 'transformer.pt')

## Validation, autoregressive decoding, and attention
The reported helper metric is teacher-forced token accuracy. Select the ordinary model explicitly with `EVAL_WEIGHTS = ASSIGNMENT_DIR / "transformer.pt"` if desired.

In [ ]:
EVAL_WEIGHTS = OUTPUT_DIR / 'transformer.pt'

In [ ]:
if RUN_EVALUATION:
    if not EVAL_WEIGHTS.is_file():
        raise FileNotFoundError(f'Train first or explicitly choose a checkpoint: {EVAL_WEIGHTS}')
    model = make_model()
    model.load_state_dict(torch.load(EVAL_WEIGHTS, map_location='cpu', weights_only=True))
    loss, accuracy = val_transformer(model, valid_loader, LabelSmoothingLoss, BATCH_SIZE, device=DEVICE)
    print('Checkpoint:', EVAL_WEIGHTS, 'validation loss:', loss, 'token accuracy:', accuracy)
    inp, inp_pos, target, target_pos = next(iter(valid_loader))
    predicted, _ = inference(model, inp[:1].to(DEVICE), inp_pos[:1].to(DEVICE), target_pos[:1].to(DEVICE), 5)
    print('Input:', ' '.join(vocab[int(i)] for i in inp[0]))
    print('Expected:', ' '.join(vocab[int(i)] for i in target[0]))
    print('Generated:', ' '.join(vocab[int(i)] for i in predicted[0]))
    custom_text = 'BOS POSITIVE 02 subtract NEGATIVE 07 EOS'
    custom = torch.tensor([prepocess_input_sequence(custom_text, token_dict, SPECIAL_TOKENS)], device=DEVICE)
    custom_pos = position_encoding_sinusoid(custom.shape[1], EMB_DIM).to(DEVICE)
    output_pos = position_encoding_sinusoid(5, EMB_DIM).to(DEVICE)
    predicted, _ = inference(model, custom, custom_pos, output_pos, 5)
    print(custom_text, '=>', ' '.join(vocab[int(i)] for i in predicted[0]))
    input_labels = [vocab[int(i)] for i in custom[0]]
    target_labels = [vocab[int(i)] for i in predicted[0, :-1]]
    for layer in range(LAYERS):
        blocks = [('encoder', model.encoder.layers[layer].MultiHeadBlock, input_labels, input_labels),
                  ('decoder_self', model.decoder.layers[layer].attention_self, target_labels, target_labels),
                  ('decoder_cross', model.decoder.layers[layer].attention_cross, input_labels, target_labels)]
        for name, block, xlabels, ylabels in blocks:
            fig, axes = plt.subplots(1, NUM_HEADS, figsize=(16, 4))
            for h, ax in enumerate(axes):
                draw(block.heads[h].weights_softmax.detach().cpu().numpy()[0], xlabels, ylabels, ax=ax)
            fig.savefig(OUTPUT_DIR / f'{name}_{layer}.jpg', bbox_inches='tight')
            plt.show()